# Data Cleaning - Battery Data Analysis
This notebook processes and cleans battery product data from an Excel file.

In [1]:
# Import Required Libraries
import pandas as pd
import numpy as np

In [2]:
# Load Excel file
# Update the file path to match your Excel file location
df = pd.read_excel('data.xlsx')  # Change path as needed
print(f"Dataset shape: {df.shape}")
print(f"\nColumn names:\n{df.columns.tolist()}")

Dataset shape: (389, 42)

Column names:
['Index', 'IndexGroup', 'Company', 'Chemistry', 'Chemistry_Detail', 'Cell_Format', 'Product_Number', 'CycleLife', 'LastCycle_ResidualCapacity_%', 'Vol_ED_WhL_Spec', 'Vol_ED_WhL_Calc', 'SpecEnergy_WhKg_Spec', 'SpecEnergy_WhKg_Calc', 'Capacity_Ah', 'OCV_Volt', 'NomVoltage_Volt', 'CutOff_Voltage_Volt', 'Discharge_Standard_A', 'Discharge_Standard_Crate', 'Discharge_MaxConstant_A', 'Discharge_MaxConstant_Crate', 'Crate_Discharge_cyclelife', 'Weight_gr', 'Volume_Calc_ccm', 'Charge_MaxConstant_A', 'Charge_MaxConstant_Crate', 'Charge_Standard_A', 'Charge_Standard_Crate', 'Crate_Charge_cyclelife', 't_rest_h', 'Length_mm', 'Thickness_Height_mm', 'Width_mm', 'Diameter_mm', 'Year', 'DataSheet', 'Secondary', 'DocumentID_Date', 'Notes', 'Charging Methode', 'Crate_CV_cutoff', 't_cycle_DOD']


In [3]:
# Filter useful columns
colonnes_utiles = [
    'Company', 'Product_Number', 'Cell_Format', 
    'Capacity_Ah', 'NomVoltage_Volt', 'Discharge_MaxConstant_A', 
    'Length_mm', 'Thickness_Height_mm', 'Width_mm', 'Diameter_mm', 
    'Weight_gr'
]
df_filtered = df[colonnes_utiles].copy()

# Display first 5 rows
print("First 5 rows of filtered data:")
df_filtered.head()

First 5 rows of filtered data:


,Company,Product_Number,Cell_Format,Capacity_Ah,NomVoltage_Volt,Discharge_MaxConstant_A,Length_mm,Thickness_Height_mm,Width_mm,Diameter_mm,Weight_gr
0,Kokam,SLPB080085270,Pouch,26.0,3.67,52,275,7.9,99,not applicable,387
1,Kokam,SLPB98188216P,Pouch,30.0,3.70,600,223,9.4,199,not applicable,780
2,Kokam,SLPB120216216,Pouch,53.0,3.70,265,227,12.0,226,not applicable,1095
3,Kokam,SLPB120216216G1,Pouch,60.0,3.68,180,227,12.0,226,not applicable,1140
4,Kokam,SLPB120216216G2,Pouch,70.0,3.67,140,227,12.3,226,not applicable,1150


## Data Cleaning Step 2: Remove Text Noise and Convert to Float
Researchers added text like "not defined", "not available", "not applicable" for missing data. FastAPI requires proper numeric values (Float). We'll replace these with NaN and convert numeric columns.

In [13]:
# 1. Replace parasitic text with proper NaN values
mots_parasites = ['not defined', 'not available', 'not applicable', '-']
df_filtered.replace(mots_parasites, np.nan, inplace=True)

# 2. Convert numeric columns to Float type
colonnes_numeriques = [
    'Capacity_Ah', 'NomVoltage_Volt', 'Discharge_MaxConstant_A', 
    'Length_mm', 'Thickness_Height_mm', 'Width_mm', 'Diameter_mm', 'Weight_gr'
]
for col in colonnes_numeriques:
    df_filtered[col] = pd.to_numeric(df_filtered[col], errors='coerce')

# 3. Remove rows with missing critical data
# Guard: Remove rows where mass (Weight_gr) is empty - required for physical model
# If we don't have capacity, voltage, max discharge current, or mass, the cell is useless
df_clean = df_filtered.dropna(subset=['Capacity_Ah', 'NomVoltage_Volt', 'Discharge_MaxConstant_A', 'Weight_gr']).copy()

print(f"Number of cells after initial cleaning: {len(df_clean)}")
print(f"Removed {len(df_filtered) - len(df_clean)} rows with missing critical values (including empty mass)")
print(f"\nNote: Row 337 in original data removed due to empty mass_g value")

Number of cells after initial cleaning: 384
Removed 5 rows with missing critical values (including empty mass)

Note: Row 337 in original data removed due to empty mass_g value


## Data Cleaning Step 3: Standardize Dimensions
Engineering step: Cylindrical cells use Diameter and Height, while Pouch cells use Length, Width, and Height. We'll create 3 universal columns: dim_L, dim_l, dim_h for API consistency.

In [5]:
def standardiser_dimensions(row):
    """Convert different cell formats to universal dimensions"""
    fmt = str(row['Cell_Format']).strip().lower()
    
    # For Cylindrical cells (diameter becomes width and length)
    if fmt == 'cylindrical':
        return pd.Series([
            row['Diameter_mm'],          # dim_L (footprint)
            row['Diameter_mm'],          # dim_l (footprint)
            row['Thickness_Height_mm']   # dim_h (height of cylinder)
        ])
    # For Pouch and Prismatic cells
    else:
        return pd.Series([
            row['Length_mm'],            # dim_L
            row['Width_mm'],             # dim_l
            row['Thickness_Height_mm']   # dim_h
        ])

# Apply standardization function
df_clean[['dim_L', 'dim_l', 'dim_h']] = df_clean.apply(standardiser_dimensions, axis=1)

# Remove rows where final dimensions couldn't be calculated
df_clean = df_clean.dropna(subset=['dim_L', 'dim_l', 'dim_h'])

print(f"Number of cells ready for 3D processing: {len(df_clean)}")
print(f"Removed {len(df_filtered) - len(df_clean)} rows with incomplete dimensions")

Number of cells ready for 3D processing: 385
Removed 4 rows with incomplete dimensions


## Data Cleaning Step 4: Add Swelling Factor (Business Rule)
The database doesn't contain swelling rates. We'll add them programmatically based on cell format, using domain knowledge from research.

In [ ]:
def calculer_swelling(format_cellule):
    """Calculate swelling percentage based on cell format"""
    fmt = str(format_cellule).strip().lower()
    
    if fmt == 'pouch':
        return 0.08          # 8% estimated swelling
    elif fmt == 'prismatic':
        return 0.03          # 3% estimated swelling
    elif fmt == 'cylindrical':
        return 0.00          # 0% swelling (rigid format)
    else:
        return 0.05          # 5% default for unknown formats

# Apply swelling calculation
df_clean['taux_swelling_pct'] = df_clean['Cell_Format'].apply(calculer_swelling)

print(f"Final dataset shape: {df_clean.shape}")
print(f"\nSwelling rates applied:")
print(df_clean['taux_swelling_pct'].value_counts())
print(f"\nDataset is now ready for API processing!")

Final dataset shape: (385, 15)

Swelling rates applied:
taux_swelling_pct
0.08    152
0.03    134
0.00     99
Name: count, dtype: int64

Dataset is now ready for API processing!


In [7]:
# Display first 5 columns and first 5 rows
print("First 5 columns of cleaned data:")
print("\nColumn names:", df_clean.columns[:5].tolist())
print("\n" + "="*80)
df_clean.iloc[:, :5].head()

First 5 columns of cleaned data:

Column names: ['Company', 'Product_Number', 'Cell_Format', 'Capacity_Ah', 'NomVoltage_Volt']



,Company,Product_Number,Cell_Format,Capacity_Ah,NomVoltage_Volt
0,Kokam,SLPB080085270,Pouch,26.0,3.67
1,Kokam,SLPB98188216P,Pouch,30.0,3.70
2,Kokam,SLPB120216216,Pouch,53.0,3.70
3,Kokam,SLPB120216216G1,Pouch,60.0,3.68
4,Kokam,SLPB120216216G2,Pouch,70.0,3.67


## Database Schema Mapping
Overview of the final database structure for battery cells with SQL types and constraints.

In [8]:
# Create database schema mapping table
schema_data = {
    'Colonne': [
        'id',
        'nom',
        'longueur_mm',
        'largeur_mm',
        'hauteur_mm',
        'masse_g',
        'tension_nominale',
        'capacite_ah',
        'courant_max_a',
        'type_cellule',
        'taux_swelling_pct'
    ],
    'Type SQL': [
        'INTEGER',
        'VARCHAR(100)',
        'FLOAT',
        'FLOAT',
        'FLOAT',
        'FLOAT',
        'FLOAT',
        'FLOAT',
        'FLOAT',
        'VARCHAR(50)',
        'FLOAT'
    ],
    'Contrainte': [
        'PK, AUTO_INCREMENT',
        'NOT NULL',
        'NOT NULL',
        'NOT NULL',
        'NOT NULL',
        'NOT NULL',
        'NOT NULL',
        'NOT NULL',
        'NOT NULL',
        'NOT NULL',
        'DEFAULT 0'
    ],
    'Description': [
        'Clé primaire auto-incrémentée',
        'Référence commerciale (ex: 18650-2500)',
        'Longueur de la cellule (mm)',
        'Largeur de la cellule (mm)',
        'Hauteur de la cellule (mm)',
        'Masse unitaire (grammes)',
        'Tension nominale Vnom (Volts)',
        'Capacité Ccell (Ampères-heure)',
        'Courant de décharge maximal Imax (A)',
        'Format/Type de cellule (Pouch, Cylindrical, Prismatic)',
        'Coefficient de gonflement (%)'
    ],
    'Source Data': [
        'Auto-generated',
        'Product_Number',
        'dim_L',
        'dim_l',
        'dim_h',
        'Weight_gr',
        'NomVoltage_Volt',
        'Capacity_Ah',
        'Discharge_MaxConstant_A',
        'Cell_Format',
        'taux_swelling_pct'
    ]
}

schema_df = pd.DataFrame(schema_data)
print("="*150)
print("DATABASE SCHEMA - BATTERY CELLS TABLE")
print("="*150)
schema_df

DATABASE SCHEMA - BATTERY CELLS TABLE


,Colonne,Type SQL,Contrainte,Description,Source Data
0,id,INTEGER,"PK, AUTO_INCREMENT",Clé primaire auto-incrémentée,Auto-generated
1,nom,VARCHAR(100),NOT NULL,Référence commerciale (ex: 18650-2500),Product_Number
2,longueur_mm,FLOAT,NOT NULL,Longueur de la cellule (mm),dim_L
3,largeur_mm,FLOAT,NOT NULL,Largeur de la cellule (mm),dim_l
4,hauteur_mm,FLOAT,NOT NULL,Hauteur de la cellule (mm),dim_h
5,masse_g,FLOAT,NOT NULL,Masse unitaire (grammes),Weight_gr
6,tension_nominale,FLOAT,NOT NULL,Tension nominale Vnom (Volts),NomVoltage_Volt
7,capacite_ah,FLOAT,NOT NULL,Capacité Ccell (Ampères-heure),Capacity_Ah
8,courant_max_a,FLOAT,NOT NULL,Courant de décharge maximal Imax (A),Discharge_MaxConstant_A
9,type_cellule,VARCHAR(50),NOT NULL,"Format/Type de cellule (Pouch, Cylindrical, Pr...",Cell_Format


## Battery Cell Types Distribution

In [9]:
# Cell type distribution
cell_types = df_clean['Cell_Format'].value_counts()

print("\nCell Types in Database:")
print("="*60)
for cell_type, count in cell_types.items():
    percentage = (count / len(df_clean)) * 100
    print(f"  {cell_type:20} : {count:4} cells ({percentage:5.1f}%)")

print("\nCell Type Characteristics:")
print("="*60)
cell_types_info = {
    'Pouch': {
        'Description': 'Flexible pouch-shaped cells',
        'Dimensions_Used': 'Length, Width, Height',
        'Swelling': '8%',
        'Rigidity': 'Flexible'
    },
    'Prismatic': {
        'Description': 'Rectangular box-shaped cells',
        'Dimensions_Used': 'Length, Width, Height',
        'Swelling': '3%',
        'Rigidity': 'Semi-rigid'
    },
    'Cylindrical': {
        'Description': 'Round cylindrical cells (e.g., 18650)',
        'Dimensions_Used': 'Diameter (L,l), Height',
        'Swelling': '0%',
        'Rigidity': 'Rigid'
    }
}

cell_types_df = pd.DataFrame(cell_types_info).T
print("\n", cell_types_df, sep='')


Cell Types in Database:
  Pouch                :  152 cells ( 39.5%)
  Prismatic            :  134 cells ( 34.8%)
  Cylindrical          :   99 cells ( 25.7%)

Cell Type Characteristics:

                                       Description         Dimensions_Used  \
Pouch                  Flexible pouch-shaped cells   Length, Width, Height   
Prismatic             Rectangular box-shaped cells   Length, Width, Height   
Cylindrical  Round cylindrical cells (e.g., 18650)  Diameter (L,l), Height   

            Swelling    Rigidity  
Pouch             8%    Flexible  
Prismatic         3%  Semi-rigid  
Cylindrical       0%       Rigid  


## Sample Data Ready for Database Import

In [10]:
# Create sample data with final column names for database
db_sample = df_clean[[
    'Product_Number', 'dim_L', 'dim_l', 'dim_h', 'Weight_gr',
    'NomVoltage_Volt', 'Capacity_Ah', 'Discharge_MaxConstant_A', 
    'Cell_Format', 'taux_swelling_pct'
]].head(10).copy()

# Rename columns to match database schema
db_sample.columns = [
    'nom', 'longueur_mm', 'largeur_mm', 'hauteur_mm', 'masse_g',
    'tension_nominale', 'capacite_ah', 'courant_max_a',
    'type_cellule', 'taux_swelling_pct'
]

print("\nSample Data for Database Table (First 10 rows):")
print("="*150)
db_sample


Sample Data for Database Table (First 10 rows):


,nom,longueur_mm,largeur_mm,hauteur_mm,masse_g,tension_nominale,capacite_ah,courant_max_a,type_cellule,taux_swelling_pct
0,SLPB080085270,275.0,99.0,7.9,387.0,3.67,26.0,52.0,Pouch,0.08
1,SLPB98188216P,223.0,199.0,9.4,780.0,3.70,30.0,600.0,Pouch,0.08
2,SLPB120216216,227.0,226.0,12.0,1095.0,3.70,53.0,265.0,Pouch,0.08
3,SLPB120216216G1,227.0,226.0,12.0,1140.0,3.68,60.0,180.0,Pouch,0.08
4,SLPB120216216G2,227.0,226.0,12.3,1150.0,3.67,70.0,140.0,Pouch,0.08
5,SLPB120255255,265.0,268.0,11.8,1535.0,3.70,75.0,225.0,Pouch,0.08
6,SLPB130255255G1,265.0,268.0,13.3,1810.0,3.70,103.0,206.0,Pouch,0.08
7,SLPB120460330,327.0,462.0,10.5,3020.0,3.70,150.0,300.0,Pouch,0.08
8,SLPB140460330,327.0,462.0,13.6,3955.0,3.70,200.0,400.0,Pouch,0.08
9,SLPB160460330,327.0,462.0,15.8,4510.0,3.70,240.0,480.0,Pouch,0.08


## Export Randomized Data to Excel

In [11]:
# Randomize the entire cleaned dataset
df_randomized = df_clean.sample(frac=1, random_state=42).reset_index(drop=True)

# Create database-ready format with randomized data
df_export = df_randomized[[
    'Product_Number', 'dim_L', 'dim_l', 'dim_h', 'Weight_gr',
    'NomVoltage_Volt', 'Capacity_Ah', 'Discharge_MaxConstant_A', 
    'Cell_Format', 'taux_swelling_pct'
]].copy()

# Rename columns to match database schema
df_export.columns = [
    'nom', 'longueur_mm', 'largeur_mm', 'hauteur_mm', 'masse_g',
    'tension_nominale', 'capacite_ah', 'courant_max_a',
    'type_cellule', 'taux_swelling_pct'
]

# Add ID column (auto-increment style)
df_export.insert(0, 'id', range(1, len(df_export) + 1))

# Export to Excel
output_file = '../Battery_Cells_Data_Randomized.xlsx'
df_export.to_excel(output_file, index=False, sheet_name='Battery_Cells')

print(f"✓ Data successfully exported to: {output_file}")
print(f"\nExport Summary:")
print(f"  Total rows exported: {len(df_export)}")
print(f"  Total columns: {len(df_export.columns)}")
print(f"  File location: {output_file}")
print(f"\nFirst 5 randomized rows:")
df_export.head()

✓ Data successfully exported to: ../Battery_Cells_Data_Randomized.xlsx

Export Summary:
  Total rows exported: 385
  Total columns: 11
  File location: ../Battery_Cells_Data_Randomized.xlsx

First 5 randomized rows:


,id,nom,longueur_mm,largeur_mm,hauteur_mm,masse_g,tension_nominale,capacite_ah,courant_max_a,type_cellule,taux_swelling_pct
0,1,LP9051109-PCM-NTC-LD,109.0,51.0,9.0,110.0,3.70,5.5,11.0,Pouch,0.08
1,2,HP-601300 NCA,60.0,60.0,159.0,980.0,3.60,27.0,270.0,Cylindrical,0.00
2,3,INR18650MJ1,18.4,18.4,65.0,49.0,3.63,3.5,10.0,Cylindrical,0.00
3,4,ICR26650,26.0,26.0,65.0,90.0,3.70,4.0,4.0,Cylindrical,0.00
4,5,CE175-360 Moxie+,253.0,172.0,5.8,430.0,3.60,17.5,17.5,Pouch,0.08


## Reference Test Dataset - 10 Most Popular Battery Cells

In [12]:
# Create 10 reference battery cells - Most popular models for testing
# Mix of Cylindrical, Pouch, and Prismatic formats

reference_cells = {
    'nom': [
        'INR18650-35E',           # Cylindrical - Samsung high capacity
        'NCR21700-50E',           # Cylindrical - Panasonic Tesla standard
        'ICR18500-1500',          # Cylindrical - Standard format
        'LP9051110-4S1P',         # Pouch - High power
        'LP103562-2S2P',          # Pouch - Energy dense
        'CE175145-3S1P',          # Pouch - Flexible pack
        'HE4-UP4',                # Cylindrical - High drain
        'LPO5063105-2S',          # Pouch - Premium
        'MP3560-75A',             # Prismatic - Power tool
        'MP4443-100A'             # Prismatic - Industrial
    ],
    'longueur_mm': [18.4, 21.0, 18.0, 175.0, 103.0, 175.0, 18.3, 149.0, 35.0, 44.0],
    'largeur_mm': [18.4, 21.0, 18.0, 51.0, 56.0, 145.0, 18.3, 56.0, 56.0, 43.0],
    'hauteur_mm': [65.0, 70.0, 50.0, 9.0, 3.5, 5.0, 65.0, 3.8, 107.5, 100.0],
    'masse_g': [48.0, 65.0, 35.0, 90.0, 32.0, 240.0, 42.0, 125.0, 180.0, 320.0],
    'tension_nominale': [3.63, 3.70, 3.70, 3.70, 3.70, 3.7, 3.6, 3.70, 3.60, 3.70],
    'capacite_ah': [3.5, 5.0, 1.5, 5.1, 4.0, 17.5, 3.0, 10.0, 7.5, 10.0],
    'courant_max_a': [10.0, 15.0, 2.0, 20.0, 8.0, 35.0, 30.0, 50.0, 75.0, 100.0],
    'type_cellule': [
        'Cylindrical',
        'Cylindrical',
        'Cylindrical',
        'Pouch',
        'Pouch',
        'Pouch',
        'Cylindrical',
        'Pouch',
        'Prismatic',
        'Prismatic'
    ],
    'taux_swelling_pct': [0.0, 0.0, 0.0, 0.08, 0.08, 0.08, 0.0, 0.08, 0.03, 0.03]
}

# Create DataFrame
df_reference = pd.DataFrame(reference_cells)
df_reference.insert(0, 'id', range(1, len(df_reference) + 1))

# Export to Excel
reference_file = '../Reference_Battery_Cells_10_Test.xlsx'
df_reference.to_excel(reference_file, index=False, sheet_name='Reference_Cells')

print("✓ Reference test dataset created successfully!")
print(f"\n📊 Export Details:")
print(f"  File: Reference_Battery_Cells_10_Test.xlsx")
print(f"  Location: {reference_file}")
print(f"  Total cells: {len(df_reference)}")
print(f"\nCell Types Distribution:")
print(f"  • Cylindrical: 4 cells (38650, 21700, 18500, HE4)")
print(f"  • Pouch: 4 cells (High power, Energy dense, Flexible pack, Premium)")
print(f"  • Prismatic: 2 cells (Power tool, Industrial)")
print(f"\n" + "="*120)
print("Reference Battery Cells for Testing:")
print("="*120)
df_reference

✓ Reference test dataset created successfully!

📊 Export Details:
  File: Reference_Battery_Cells_10_Test.xlsx
  Location: ../Reference_Battery_Cells_10_Test.xlsx
  Total cells: 10

Cell Types Distribution:
  • Cylindrical: 4 cells (38650, 21700, 18500, HE4)
  • Pouch: 4 cells (High power, Energy dense, Flexible pack, Premium)
  • Prismatic: 2 cells (Power tool, Industrial)

Reference Battery Cells for Testing:


,id,nom,longueur_mm,largeur_mm,hauteur_mm,masse_g,tension_nominale,capacite_ah,courant_max_a,type_cellule,taux_swelling_pct
0,1,INR18650-35E,18.4,18.4,65.0,48.0,3.63,3.5,10.0,Cylindrical,0.00
1,2,NCR21700-50E,21.0,21.0,70.0,65.0,3.70,5.0,15.0,Cylindrical,0.00
2,3,ICR18500-1500,18.0,18.0,50.0,35.0,3.70,1.5,2.0,Cylindrical,0.00
3,4,LP9051110-4S1P,175.0,51.0,9.0,90.0,3.70,5.1,20.0,Pouch,0.08
4,5,LP103562-2S2P,103.0,56.0,3.5,32.0,3.70,4.0,8.0,Pouch,0.08
5,6,CE175145-3S1P,175.0,145.0,5.0,240.0,3.70,17.5,35.0,Pouch,0.08
6,7,HE4-UP4,18.3,18.3,65.0,42.0,3.60,3.0,30.0,Cylindrical,0.00
7,8,LPO5063105-2S,149.0,56.0,3.8,125.0,3.70,10.0,50.0,Pouch,0.08
8,9,MP3560-75A,35.0,56.0,107.5,180.0,3.60,7.5,75.0,Prismatic,0.03
9,10,MP4443-100A,44.0,43.0,100.0,320.0,3.70,10.0,100.0,Prismatic,0.03
